# TCD DAT → Parquet 변환
> 교통카드 데이터(TCD, STTN, ROUTE, ROUTESTTN) DAT 파일을 Parquet 포맷으로 일괄 변환
> - 입력: `C:/Folder/Research/0. DATA/tcd_2025/DATA_{date}/` 내 DAT 파일
> - 출력: `C:/Folder/Research/0. DATA/tcd_2025_parquet/{date}/` 내 Parquet 파일
> - 대상: TCD(이용내역), STTN(정류장), ROUTE(노선), ROUTESTTN(노선-정류장)

---
## 1. 설정

In [1]:
import pandas as pd
import os

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

from loguru import logger

---
## 2. DAT → Parquet 변환 클래스 정의

In [2]:
def _read_dat(filepath, columns, sep='|', dtype=None):
    for enc in ('cp949', 'utf-8'):
        try:
            return pd.read_csv(filepath, encoding=enc, sep=sep, names=columns, dtype=dtype).reset_index(drop=True)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"cp949/utf-8 모두 실패: {filepath}")


class MultiDateBusDataProcessor:
    TCD_COLUMNS = [
        '운행일자', '정산사 ID', '인련번호', '가상카드번호', '정산지역코드', '카드구분코드',
        '차량ID(국토부표준)', '차량ID(정산사업자)', '차량등록번호', '운행출발일시', '운행종료일시',
        '교통수단코드', '노선ID(국토부표준)', '노선ID(정산사업자)', '승차일시', '발권일시',
        '승차정류장ID(국토부표준)', '승차정류장ID(정산사업자)', '하차정류장ID(국토부표준)',
        '하차정류장ID(정산사업자)', '하차일시', '트랜잭션ID', '환승건수', '사용자구분코드', '이용자수',
        '이용거리', '탑승시간']

    STTN_COLUMNS = [
        '운행일자', '정산사 ID', '정산지역코드', '정류장 ID', '정류장 명칭', '정류장 ARS번호',
        '정류장 X 좌표', '정류장 Y 좌표', '시도코드', '시도명', '시군구코드', '시군구명', '읍면동코드', '읍면동명']

    ROUTE_COLUMNS = [
        '운행일자', '정산사 ID', '정산지역코드', '노선ID', '노선명(long)', '노선명(short)',
        '교통수단유형', '총운행거리', '정류장수']

    ROUTESTTN_COLUMNS = [
        '운행일자', '정산사 ID', '정산지역코드', '노선ID', '노선명(short)', '교통수단유형', '정류장순서',
        '정류장 ID', '정류장 명칭', '정류장 X 좌표', '정류장 Y 좌표', '정류장 ARS번호', '누적거리(m)', '구간거리(m)']

    # 파일별 설정: (dat파일명, 컬럼, parquet파일명, astype 변환할 컬럼들, read_csv dtype 지정)
    FILE_CONFIG = {
        'TCD': ('DWTCD_{date}.dat', 'TCD_COLUMNS', 'TCD_{date}.parquet',
                ['정산지역코드', '카드구분코드', '노선ID(정산사업자)'], {}),
        'STTN': ('STTN_{date}.dat', 'STTN_COLUMNS', 'STTN_{date}.parquet', [], {}),
        'ROUTE': ('ROUTE_{date}.dat', 'ROUTE_COLUMNS', 'ROUTE_{date}.parquet',
                  ['정산지역코드'], {'노선ID': str}),
        'ROUTESTTN': ('ROUTESTTN_{date}.dat', 'ROUTESTTN_COLUMNS', 'ROUTESTTN_{date}.parquet',
                      ['정산지역코드'], {'노선ID': str}),
    }

    def __init__(self, start_date, end_date, targets=None):
        self.start_date = pd.to_datetime(start_date)
        self.end_date = pd.to_datetime(end_date)
        self.targets = targets or list(self.FILE_CONFIG.keys())

    def process_date(self, date_str):
        base_path = f"C:/Folder/Research/0. DATA/tcd_2025/DATA_{date_str}/"
        date_folder = f"C:/Folder/Research/0. DATA/tcd_2025_parquet/{date_str}/"
        os.makedirs(date_folder, exist_ok=True)

        for name in self.targets:
            dat_name, col_attr, pq_name, str_cols, read_dtype = self.FILE_CONFIG[name]
            logger.info(f'load_{name}')
            df = _read_dat(
                os.path.join(base_path, dat_name.format(date=date_str)),
                getattr(self, col_attr),
                dtype=read_dtype
            )
            for col in str_cols:
                df[col] = df[col].astype(str)
            if name == 'ROUTE':
                gtx_mask = df['노선명(long)'].str.contains('gtx', case=False, na=False)
                df.loc[gtx_mask, '교통수단유형'] = 'G'
                
            if name == 'ROUTESTTN':
                gtx_mask = df['노선명(short)'].str.contains('gtx', case=False, na=False)
                df.loc[gtx_mask, '교통수단유형'] = 'G'
                
            df.to_parquet(os.path.join(date_folder, pq_name.format(date=date_str)), index=False)
            logger.info(f'save_{name}_finish')

    def process_all_dates(self):
        for single_date in tqdm(pd.date_range(start=self.start_date, end=self.end_date)):
            date_str = single_date.strftime('%Y%m%d')
            self.process_date(date_str)

---
## 3. 실행

In [3]:
# ROUTE, ROUTESTTN만 처리
processor = MultiDateBusDataProcessor('20250217', '20250223', targets=['ROUTE', 'ROUTESTTN'])
processor.process_all_dates()

  0%|          | 0/7 [00:00<?, ?it/s]2026-02-04 18:12:46.687 | INFO     | __main__:process_date:54 - load_ROUTE
2026-02-04 18:12:46.735 | INFO     | __main__:process_date:71 - save_ROUTE_finish
2026-02-04 18:12:46.736 | INFO     | __main__:process_date:54 - load_ROUTESTTN
2026-02-04 18:12:47.121 | INFO     | __main__:process_date:71 - save_ROUTESTTN_finish
 14%|█▍        | 1/7 [00:00<00:02,  2.29it/s]2026-02-04 18:12:47.124 | INFO     | __main__:process_date:54 - load_ROUTE
2026-02-04 18:12:47.143 | INFO     | __main__:process_date:71 - save_ROUTE_finish
2026-02-04 18:12:47.143 | INFO     | __main__:process_date:54 - load_ROUTESTTN
2026-02-04 18:12:47.523 | INFO     | __main__:process_date:71 - save_ROUTESTTN_finish
 29%|██▊       | 2/7 [00:00<00:02,  2.40it/s]2026-02-04 18:12:47.527 | INFO     | __main__:process_date:54 - load_ROUTE
2026-02-04 18:12:47.544 | INFO     | __main__:process_date:71 - save_ROUTE_finish
2026-02-04 18:12:47.545 | INFO     | __main__:process_date:54 - load_ROU